In [ ]:
import sys
import os
import glob
import numpy as np
import pandas as pd
import plotnine as pn
import patchworklib as pw

import scipy
import statsmodels.formula.api as smf
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances

sys.path.append(os.path.dirname(os.path.abspath('')))
import scripts.dataloader as dataloader

In [ ]:
def filter_genes_for_annotation(gene_set):
    return list(pd.merge(pd.Series(gene_set).to_frame("gene_id"), pd.Series(pd.read_parquet("/s/project/uk_biobank/processed/derived_datasets/ukbb_wes_200k_ensembl_loftee_plof_binarized_pivot.parquet").columns).to_frame("gene_id"))["gene_id"].unique())

In [ ]:
def filter_genes_for_embedding(gene_set, embedding_type="omics_pops"):
    return list(pd.merge(pd.Series(gene_set).to_frame("gene_id"), pd.read_csv(dataloader.EMBEDDINGS[embedding_type], sep="\t")[["gene_id"]])["gene_id"].unique())

In [ ]:
def get_gene_neighbourhood(index_gene_id, embedding_type="omics_pops", neighbourhood_size=50):
    gene_embeddings_df = pd.read_csv(dataloader.EMBEDDINGS[embedding_type], sep="\t").set_index("gene_id")
    pairwise_dist = euclidean_distances(gene_embeddings_df)
    genes_by_distance = np.argsort(pairwise_dist[gene_embeddings_df.index.get_loc(index_gene_id)])
    gene_distance_rank = gene_embeddings_df.iloc[genes_by_distance].reset_index()[["gene_id"]]
    gene_distance_rank = gene_distance_rank.rename_axis('distance_rank')
    return gene_distance_rank.head(neighbourhood_size)["gene_id"].to_list()

In [ ]:
def plot_residual_of_pLoF_carriers(trait, gene_set, dataset_df, splits=["train", "val", "test"]):
    dataset_df = dataset_df[dataset_df[f"{trait}_split"].isin(splits)]
    pLoF_df = pd.read_parquet("/s/project/uk_biobank/processed/derived_datasets/ukbb_wes_200k_ensembl_loftee_plof_binarized_pivot.parquet")
    pLoF_df = pLoF_df[gene_set].reset_index().melt(id_vars='individual', value_name="carrier").query("carrier")
    plot_df = pd.merge(dataset_df, pLoF_df)
    plot_df["Known association"] = plot_df["gene"].isin(replication_set)
    # plot_df["gene_id"] = pd.Categorical(plot_df["gene"], ordered=True,  categories=gene_set)

    hgnc2ensg = pd.read_csv('/s/project/geno2pheno/data/hgnc2ensg.tsv', sep='\t')
    hgnc2ensg.rename(columns={'Ensembl gene ID':'gene', 'Approved symbol':'gene_symbol'}, inplace=True)
    plot_df = pd.merge(plot_df, hgnc2ensg[['gene', 'gene_symbol']], on='gene')
    
    plot_df["gene_id"] = pd.Categorical(plot_df["gene_symbol"], ordered=True,  categories=[hgnc2ensg[hgnc2ensg.gene==g]['gene_symbol'].item() for g in gene_set])
    plot_df["gene_id"] = plot_df["gene_id"].cat.reorder_categories(plot_df["gene_id"].cat.categories[::-1])
    
    sample_sizes = plot_df.groupby("gene_id").size().reset_index()
    sample_sizes.columns = ['gene_id', 'n_carriers']
    
    return plot_df, sample_sizes

In [ ]:
trait = 'HDL_cholesterol' #"Apolipoprotein_A" "ENSG00000165029"

DATA_BASEPATH = "/s/project/uk_biobank/processed/derived_datasets/"
dataset_df = pd.read_parquet(f"{DATA_BASEPATH}ukbb_common_traits_and_covariates_with_split_and_residuals_filteredv3.parquet")
dataset_df = dataset_df[dataset_df.columns[:23].to_list()+[c for c in dataset_df.columns if trait in c]].dropna().reset_index(drop=True)
phenocode_dict = dataloader.load_phenocode_dict()
replication_set = dataloader.load_replication_set([trait], key="trait")[trait]

In [ ]:
%%time
gene_set = get_gene_neighbourhood("ENSG00000087237", embedding_type="omics_pops", neighbourhood_size=25)
gene_set = filter_genes_for_embedding(filter_genes_for_annotation(gene_set), embedding_type="omics_pops")

plot_df, sample_sizes = plot_residual_of_pLoF_carriers(trait, gene_set, dataset_df, ["train", "val"])

dt = pd.read_parquet('/s/project/geno2pheno/predictions/bayesian/best_model_pred/v108cov_deepRVAT_genes_extended.pq')

sub_dt = dt[(dt.trait==trait) & (dt.gene_id.isin(gene_set))]

hgnc2ensg = pd.read_csv('/s/project/geno2pheno/data/hgnc2ensg.tsv', sep='\t')
hgnc2ensg.rename(columns={'Ensembl gene ID':'gene_id', 'Approved symbol':'gene_symbol'}, inplace=True)
sub_dt = pd.merge(sub_dt, hgnc2ensg[['gene_id', 'gene_symbol']], on='gene_id')
sub_dt

sub_dt["gene_id"] = pd.Categorical(sub_dt["gene_symbol"], ordered=True,  categories=[hgnc2ensg[hgnc2ensg.gene_id==g]['gene_symbol'].item() for g in gene_set])
sub_dt["gene_id"] = sub_dt["gene_id"].cat.reorder_categories(sub_dt["gene_id"].cat.categories)
plot_df["gene_id"] = plot_df["gene_id"].cat.reorder_categories(sub_dt["gene_id"].cat.categories)
# sub_dt["gene_id"] = sub_dt["gene_id"].cat.reorder_categories(sub_dt["gene_id"].cat.categories[::-1])

# sub_dt["gene_id"] = pd.Categorical(sub_dt["gene_symbol"], ordered=True,  categories=[g for g in plot_df.gene_id.cat.categories])
sub_dt['neg_fE'] = -sub_dt['fE']
sub_dt.sort_values('gene_id', inplace=True, ascending=True)
sub_dt.reset_index(drop=True, inplace=True)
sub_dt['id'] = sub_dt.index + 1
sub_dt.head()

In [ ]:
N = dt[(dt.trait=='HDL_cholesterol')].shape[0]
M = dt[(dt.trait=='HDL_cholesterol') & (dt.replicated == True)].shape[0] - 1
k = 24
n = 4

enr = (n*N)/(k*M)
enr

In [ ]:
w_size = 1
sub_dt['roll_fE'] = sub_dt['fE'].rolling(window=w_size, min_periods=1).mean()
sub_dt['roll_neg_fE'] = sub_dt['neg_fE'].rolling(window=w_size, min_periods=1).mean()
sub_dt

plot = (pn.ggplot()
        + pn.geoms.geom_hline(yintercept = 0)
        + pn.geom_text(sample_sizes, pn.aes(x='gene_id', y=plot_df[f"{trait}_common_resid"].min()-0.5, label='n_carriers'), size=8, va='top')
        # + pn.geom_line(sub_dt, pn.aes(x='id', y='beta'), linetype='dashed')
        + pn.geom_ribbon(sub_dt, pn.aes(x='id', ymin='roll_neg_fE', ymax='roll_fE'), alpha=0.25)
        + pn.geom_boxplot(plot_df, pn.aes(x='gene_id', y=f"{trait}_common_resid", fill="Known association"), width=0.5)
        + pn.theme_light()
        + pn.ylab(f"{trait}\n(adjusted)")
        + pn.xlab("CETP embedding neighbours\n(ordered by distance)")
        + pn.scales.scale_fill_manual(values=["grey", "green"])
        + pn.theme(legend_position = 'bottom',
                   legend_box = 'horizontal',
                   figure_size=(8, 4.5),
                   axis_text_x = pn.element_text(angle = 90),
                   text=pn.element_text(size=10))
        # + pn.coord_flip()
    )

plot

In [ ]:
# SAVE figure

pn.ggsave(plot, '/s/project/geno2pheno/figures/GeneSet25_CETP_fE.svg', format='svg', dpi=300)
# pn.ggsave(plot, '/s/project/geno2pheno/figures/GeneSet_ABCA1_fE.svg', format='svg', dpi=300)


# More plots

In [ ]:
def get_top_effect_genes(n=50, min_carriers=10):
    mean_pheno_pLoF_df = dataloader.load_common_residuals_statistics_by_split(trait)
    mean_pheno_pLoF_df["mean_pheno_abs"] = abs(mean_pheno_pLoF_df["mean_pheno"])
    return list(mean_pheno_pLoF_df.query(f"n_carriers>={min_carriers}").sort_values("mean_pheno_abs", ascending=False).head(n).index)

In [ ]:
def get_top_fE_genes(bayes_version, n=50):
    base_dir = "/s/project/geno2pheno/predictions/bayesian"
    bayes_fE_df = pd.read_parquet(f"{base_dir}/{bayes_version}_genes.pq")
    bayes_fE_df = bayes_fE_df.query(f"trait=='{trait}' and model=='omics_pops_bayesian_{bayes_version}'").reset_index(drop=True)
    bayes_fE_df = bayes_fE_df.sort_values("fE", ascending=False)
    return list(bayes_fE_df.head(n)["gene_id"])

In [ ]:
def plot_model_ci(gene_set, bayes_versions):
    base_dir = "/s/project/geno2pheno/predictions/bayesian"
    genotype = "pLoF"
    ols_pLoF_df = pd.read_parquet(f"/s/project/uk_biobank/processed/g2p/modelling/ols_{genotype}/filteredv2_ols_{genotype}_cov_full_{trait}.pq").reset_index()
    ols_pLoF_df["model"] = "LM" 

    bayes_betas = []
    for version in bayes_versions:
        bayes_betas_df = pd.read_parquet(f"{base_dir}/{version}_genes.pq")
        bayes_betas_df = bayes_betas_df.query(f"trait=='{trait}' and model=='omics_pops_bayesian_{version}'").reset_index(drop=True)
        bayes_betas_df["model"] = f"funcRVAT_{version}"
        bayes_betas.append(bayes_betas_df[["gene_id", "beta", "std_err", "model"]])

    plot_ci_df = pd.concat(bayes_betas + [ols_pLoF_df.rename(columns={"index": "gene_id"})[["gene_id", "beta", "std_err", "model"]]], axis=0)

    confidence_level = 0.95
    alpha = 1 - confidence_level
    critical_value = scipy.stats.norm.ppf(1 - alpha / 2)
    multiplied_values = plot_ci_df["std_err"] * critical_value
    plot_ci_df["lower_bound"] = plot_ci_df["beta"] - multiplied_values
    plot_ci_df["upper_bound"] = plot_ci_df["beta"] + multiplied_values

    plot_ci_df = plot_ci_df[plot_ci_df["gene_id"].isin(gene_set)].reset_index(drop=True)
    plot_ci_df["gene_id"] = pd.Categorical(plot_ci_df["gene_id"], ordered=True,  categories=gene_set)
    
    plot = (
        pn.ggplot(plot_ci_df, pn.aes(x='gene_id', y="beta", color="model"))
        + pn.geoms.geom_hline(yintercept = 0)
        + pn.geom_point(position = pn.position_dodge(width = 0.5, preserve = "single"), stroke = 0, size=2)
        + pn.geom_errorbar(pn.aes(ymin = "lower_bound", ymax = "upper_bound"), position = pn.position_dodge(width = 0.5, preserve = "single"), size = 1.5)
        + pn.theme_light()
        #+ pn.ylab(y_axis_name)
        + pn.theme(figure_size=(20,5), axis_text_x=pn.element_blank(), axis_title_x=pn.element_blank(), text=pn.element_text(size=12))
        #+ pn.labels.ggtitle(f"Credible interval of model betas for the genes from above")
    )
    return plot

In [ ]:
def plot_model_fE(gene_set, bayes_versions):
    base_dir = "/s/project/geno2pheno/predictions/bayesian"

    bayes_fE = []
    for version in bayes_versions:
        bayes_fE_df = pd.read_parquet(f"{base_dir}/{version}_genes.pq")
        bayes_fE_df = bayes_fE_df.query(f"trait=='{trait}' and model=='omics_pops_bayesian_{version}'").reset_index(drop=True)
        bayes_fE_df = bayes_fE_df[bayes_fE_df["gene_id"].isin(gene_set)]
        bayes_fE_df["model"] = f"funcRVAT_{version}"
        bayes_fE.append(bayes_fE_df[["gene_id", "fE", "model"]])

    plot_fE_df = pd.concat(bayes_fE)
    plot_fE_df = pd.concat([plot_fE_df, pd.DataFrame({"model": "LM", "gene_id": list(bayes_fE_df["gene_id"].unique())})])
    plot_fE_df["gene_id"] = pd.Categorical(plot_fE_df["gene_id"], ordered=True, categories=gene_set)
    
    plot = (
        pn.ggplot(plot_fE_df)
        + pn.geom_bar(pn.aes(y="fE", x="gene_id", fill="model"), stat='identity', position='dodge')
        #+ pn.geom_line(pn.aes(y="fE", x="gene_id", group="model", color="model"))
        + pn.theme_light()
        #+ pn.ylab(y_axis_name)
        + pn.theme(figure_size=(20,5), axis_text_x=pn.element_text(rotation=90), text=pn.element_text(size=12))
        #+ pn.labels.ggtitle(f"Credible interval of model betas for the genes from above")
    )
    return plot

In [ ]:
genebass_set = list(pd.read_parquet("/s/project/rep/processed/genebass/500k/filtered.parquet").query(f"phenocode=='{phenocode_dict[trait]}'")[["phenocode", "description", "gene_id", "BETA_Burden", "SE_Burden", "Pvalue_Burden"]]["gene_id"].unique())

In [ ]:
gene_neighbourhood = get_gene_neighbourhood("ENSG00000166165", neighbourhood_size=50)

In [ ]:
top_effect_genes = get_top_effect_genes(n=50, min_carriers=10)

In [ ]:
top_fE_genes = get_top_fE_genes("v83cov_pLoF")

In [ ]:
%%time
gene_set = genebass_set
gene_set = filter_genes_for_embedding(filter_genes_for_annotation(gene_set))
funcRVAT_versions = ["v83cov_pLoF", "v84cov_pLoF"]
figure_size = (20,5)
g1 = pw.load_ggplot(plot_residual_of_pLoF_carriers(gene_set, dataset_df, ["train", "val"]), figsize=figure_size)
g2 = pw.load_ggplot(plot_model_ci(gene_set, funcRVAT_versions), figsize=figure_size)
g3 = pw.load_ggplot(plot_model_fE(gene_set, funcRVAT_versions), figsize=figure_size)
g123_v = (g1/g2)/g3
g123_v

In [ ]:
gene_embeddings_df = pd.read_csv(dataloader.EMBEDDINGS["omics_pops"], sep="\t").set_index("gene_id")
labels = gene_neighbourhood[:4]
pca = PCA(n_components=2)
embeddings_pcs = pca.fit_transform(gene_embeddings_df.values)
pc_df = pd.DataFrame(embeddings_pcs, index=gene_embeddings_df.index, columns=["pc0", "pc1"]).loc[gene_neighbourhood].reset_index(names=["gene_id"])
pc_df["label"] = pc_df.apply(lambda g: g["gene_id"] if g["gene_id"] in labels else "", axis=1)
plot = (
        pn.ggplot(pc_df)
        + pn.geom_point(pn.aes(y="pc1", x="pc0"))
        + pn.geom_text(pn.aes(y="pc1", x="pc0", label='label'), nudge_y=1, nudge_x=0, size=10)
        + pn.theme_light()
        + pn.theme(figure_size=(16,16))
        #+ pn.labels.ggtitle(f"Credible interval of model betas for the genes from above")
)
plot

In [ ]:
base_dir = "/s/project/geno2pheno/predictions/bayesian"
bayes_version = "v84cov_pLoF"
bayes_fE_df = pd.read_parquet(f"{base_dir}/{bayes_version}_genes.pq")
bayes_fE_df = bayes_fE_df.query(f"trait=='{trait}' and model=='omics_pops_bayesian_{bayes_version}'").reset_index(drop=True)
bayes_fE_df = bayes_fE_df.sort_values("fE", ascending=False)

In [ ]:
mean_pheno_pLoF_df = dataloader.load_common_residuals_statistics_by_split(trait).query("n_carriers<10_000")
mean_pheno_pLoF_df["n_carriers_trainval"] = mean_pheno_pLoF_df["n_carriers_train"] + mean_pheno_pLoF_df["n_carriers_val"]

In [ ]:
pd.merge(bayes_fE_df, mean_pheno_pLoF_df.reset_index(names=["gene_id"]))

In [ ]:
plot = (
        pn.ggplot(pd.merge(bayes_fE_df, mean_pheno_pLoF_df.reset_index(names=["gene_id"])))
        + pn.geom_bin_2d(pn.aes(y="fE", x="n_carriers+1"), bins=100)
        + pn.theme_light()
        + pn.theme(figure_size=(6,6))
        + pn.scales.scale_x_log10()
        + pn.scales.scale_y_log10()
        #+ pn.labels.ggtitle(f"Credible interval of model betas for the genes from above")
)
plot